# K-SPADA Data Preparation

[Project overview](../README.md) · [01 Data preparation](./01_data_preparation.ipynb) · [02 Model training](./02_model_training.ipynb) · [03 Prediction and visualization](./03_prediction_and_visualization.ipynb) · [04 Quantile binning](./04_binning_data_preparation.ipynb)

This tutorial prepares training, validation, and query communication-graph files from a generic spatial transcriptomics AnnData object.

## Workflow
1. Define paths and sample splits.
2. Validate the required AnnData fields.
3. Separate reference and query samples.
4. Build pathway features and communication graphs.

Run this notebook from the `tutorials/` directory. Paths are relative to this directory or to the repository root (`..`). This example uses samples 4–15 for training, sample 16 for validation, and samples 17–21 for queries. The source dataset is stored at `../../../../TOSICA-main/data/liver_cancer/all_hvg_3000.h5ad`; adapt `DATA_DIR` to your own data location.


In [1]:
from pathlib import Path
import anndata as ad
import KSPADA

# PROJECT_ROOT = Path("..").resolve()
# DATA_DIR = PROJECT_ROOT / "experiment_data"
# OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_preparation"
# SOURCE_H5AD = DATA_DIR / "source_data.h5ad"

# TRAIN_SAMPLES = ["train_sample_1", "train_sample_2"]
# VALIDATION_SAMPLES = ["validation_sample"]
# QUERY_SAMPLES = ["query_sample"]
COMMUNICATION_GRAPH_NEIGHBORS = 2
FORCE_OVERWRITE = True

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = Path("../../../../TOSICA-main/data/liver_cancer").resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "liver_cancer_data_preparation"
SOURCE_H5AD = DATA_DIR / "all_hvg_3000.h5ad"

TRAIN_SAMPLES = ["4", "5", "6", "7", "8", "9", "10", "11",
                 "12", "13", "14", "15"]
VALIDATION_SAMPLES = ["16"]
QUERY_SAMPLES = ["17", "18", "19", "20", "21"]

## Step 1 — Validate the source data

The source AnnData object must contain spatial coordinates, sample identifiers, and training labels.

In [2]:
source_adata = ad.read_h5ad(SOURCE_H5AD)
required_columns = {"x", "y", "batch", "Celltype"}
missing_columns = required_columns.difference(source_adata.obs.columns)
if missing_columns:
    raise KeyError(f"Missing required obs columns: {sorted(missing_columns)}")

source_adata.obs["batch"] = source_adata.obs["batch"].astype(str)
requested_samples = set(TRAIN_SAMPLES + VALIDATION_SAMPLES + QUERY_SAMPLES)
missing_samples = requested_samples.difference(source_adata.obs["batch"].unique())
if missing_samples:
    raise ValueError(f"Missing requested samples: {sorted(missing_samples)}")

/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## Step 2 — Split reference and query data

In [3]:
train_valid_adata = source_adata[
    source_adata.obs["batch"].isin(TRAIN_SAMPLES + VALIDATION_SAMPLES)
].copy()
query_adata = source_adata[source_adata.obs["batch"].isin(QUERY_SAMPLES)].copy()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_valid_source = OUTPUT_DIR / "reference_source.h5ad"
query_source = OUTPUT_DIR / "query_source.h5ad"
KSPADA.write_adata(train_valid_adata, train_valid_source, force=FORCE_OVERWRITE)
KSPADA.write_adata(query_adata, query_source, force=FORCE_OVERWRITE)

/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


PosixPath('/home/lixiangyu/sh/program/K-SPADA/outputs/liver_cancer_data_preparation/query_source.h5ad')

## Step 3 — Build pathway features and communication graphs

In [ ]:
prepared_paths = KSPADA.prepare_data_workflow(
    train_h5ad=train_valid_source,
    query_h5ad=query_source,
    output_dir=OUTPUT_DIR,
    train_batches=TRAIN_SAMPLES,
    valid_batches=VALIDATION_SAMPLES,
    query_batches=QUERY_SAMPLES,
    cgraph_k=COMMUNICATION_GRAPH_NEIGHBORS,
    train_output_name="reference_data",
    query_output_name="query_data",
    force=FORCE_OVERWRITE,
)

/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/an

## Notes

- Keep biological samples separate when assigning training, validation, and query data.
- Replace the sample names with values present in `adata.obs["batch"]`.

The configuration now matches the training/validation split used in tutorials 02–03. This notebook was not rerun on 2026-11-10; its earlier preparation outputs are retained. For the executed training example, the existing per-sample graphs were merged again with samples 4–15 for training and sample 16 for validation.
